# Seasonal Agriculture Performance Analysis

**VOIS AICTE Batch 1, 2026-2027 — Major Project**

This notebook analyzes how agricultural performance (yield, production, resource usage, and economics) varies across the three Indian cropping seasons — **Kharif, Rabi and Zaid** — using the provided farm-level dataset.

**Structure of this notebook**
1. Setup & Data Loading
2. Initial Data Exploration
3. Data Cleaning & Preparation
4. Univariate Analysis
5. Seasonal Performance Comparison (Yield & Production)
6. Environmental Conditions Across Seasons
7. Resource Usage Across Seasons
8. Relationships Between Environmental Conditions and Performance
9. Economic Outcomes Across Seasons
10. Regional & Crop-wise Consistency of Seasonal Patterns
11. Statistical Hypothesis Testing
12. Unusual / Unexpected Pattern Detection
13. Key Insights Summary
14. Conclusions & Recommendations

Each section maps directly to the key questions in the project brief.

## 1. Setup & Data Loading

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 13
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

SEASON_ORDER = ['Kharif', 'Rabi', 'Zaid']   # fixed, meaningful order for all plots
np.random.seed(42)

**Loading the dataset in Colab** — run ONE of the two options below.

* **Option A (recommended):** Upload the CSV directly using the file picker that appears when you run the cell.
* **Option B:** If you've already placed the file in your Colab session (e.g. via Google Drive), just set `DATA_PATH` to that location and skip the upload cell.

In [ ]:
# Option A: Upload the CSV from your computer (uncomment to use)
# from google.colab import files
# uploaded = files.upload()
# DATA_PATH = list(uploaded.keys())[0]

# Option B: File already present in the Colab working directory / Drive
DATA_PATH = 'seasonal_agriculture_performance_dataset.csv'

df = pd.read_csv(DATA_PATH)
df['Season'] = pd.Categorical(df['Season'], categories=SEASON_ORDER, ordered=True)
print("Dataset loaded:", df.shape[0], "rows,", df.shape[1], "columns")
df.head()

## 2. Initial Data Exploration

Before analyzing seasonal patterns, we first understand the structure, data types and coverage of the dataset.

In [ ]:
df.info()

In [ ]:
print("Categorical fields:\n")
for col in ['State', 'District', 'Crop', 'Season', 'Irrigation_Method']:
    print(f"{col} ({df[col].nunique()} unique): {sorted(df[col].astype(str).unique())}")

In [ ]:
df.describe().T

In [ ]:
print("Rows per season:")
print(df['Season'].value_counts().reindex(SEASON_ORDER))
print("\nRows per season, as % of total:")
print((df['Season'].value_counts(normalize=True).reindex(SEASON_ORDER) * 100).round(1))

## 3. Data Cleaning & Preparation

### 3.1 Missing values
We check for missing values and treat them **season-wise** (using the season's own median) rather than a single global median, since rainfall, soil moisture and yield are all strongly season-dependent — imputing with a global value would blur real seasonal differences.

In [ ]:
missing = df.isnull().sum()
print("Columns with missing values:")
print(missing[missing > 0])

In [ ]:
for col in ['Rainfall_mm', 'Soil_Moisture_pct', 'Yield_Tonnes_Ha']:
    before = df[col].isnull().sum()
    df[col] = df.groupby('Season', observed=True)[col].transform(lambda x: x.fillna(x.median()))
    print(f"{col}: filled {before} missing values using season-wise median")

print("\nRemaining missing values in dataset:", df.isnull().sum().sum())

### 3.2 Duplicate records

In [ ]:
dupes = df.duplicated().sum()
print("Duplicate rows found:", dupes)
df = df.drop_duplicates().reset_index(drop=True)

### 3.3 Feature engineering

We derive a few additional fields that make the economic and comparative analysis easier:

- **Profit_Margin_pct** — profit as a percentage of revenue (comparable across farms of different sizes)
- **Cost_per_Tonne_INR** — cost efficiency per tonne produced
- **Is_Loss_Making** — flag for farms operating at a loss
- **Yield_Zscore_within_Crop** — yield standardized *within each crop*. Raw `Yield_Tonnes_Ha` isn't directly comparable across crops (e.g. Sugarcane naturally yields 40-100+ t/ha, while Wheat yields 2-5 t/ha) — this normalized version lets us compare *relative* performance across seasons without crop-mix bias.

In [ ]:
df['Profit_Margin_pct'] = (df['Profit_INR'] / df['Revenue_INR']) * 100
df['Cost_per_Tonne_INR'] = df['Total_Cost_INR'] / df['Production_Tonnes']
df['Is_Loss_Making'] = df['Profit_INR'] < 0
df['Yield_Zscore_within_Crop'] = df.groupby('Crop')['Yield_Tonnes_Ha'].transform(
    lambda x: (x - x.mean()) / x.std()
)

df[['Profit_Margin_pct', 'Cost_per_Tonne_INR', 'Is_Loss_Making', 'Yield_Zscore_within_Crop']].describe(include='all').T

### 3.4 Outlier check

We flag statistical outliers in `Yield_Tonnes_Ha` using the IQR method. We **flag rather than delete** them at this stage — as shown below, most "outliers" are simply Sugarcane, which naturally has a much higher yield per hectare than grains, so they represent real agronomic variation, not data errors.

In [ ]:
Q1, Q3 = df['Yield_Tonnes_Ha'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
df['Yield_Outlier'] = ~df['Yield_Tonnes_Ha'].between(lower, upper)

print(f"Yield outlier bounds: [{lower:.2f}, {upper:.2f}] tonnes/ha")
print("Total flagged outliers:", df['Yield_Outlier'].sum())
print("\nCrop composition of flagged outliers:")
print(df.loc[df['Yield_Outlier'], 'Crop'].value_counts())

## 4. Univariate Analysis

A quick look at how categories and key numeric fields are distributed before comparing across seasons.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x='Season', order=SEASON_ORDER, hue='Season', palette='Set2', legend=False, ax=axes[0,0])
axes[0,0].set_title('Records by Season')

sns.countplot(data=df, y='Crop', order=df['Crop'].value_counts().index, hue='Crop', palette='Set3', legend=False, ax=axes[0,1])
axes[0,1].set_title('Records by Crop')

sns.countplot(data=df, y='State', order=df['State'].value_counts().index, hue='State', palette='Set3', legend=False, ax=axes[1,0])
axes[1,0].set_title('Records by State')

sns.countplot(data=df, x='Irrigation_Method', order=df['Irrigation_Method'].value_counts().index,
              hue='Irrigation_Method', palette='Set2', legend=False, ax=axes[1,1])
axes[1,1].set_title('Records by Irrigation Method')

fig.tight_layout()
plt.show()

In [ ]:
key_metrics = ['Yield_Tonnes_Ha', 'Profit_INR', 'Rainfall_mm', 'Water_Efficiency_t_per_1000m3']

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, key_metrics):
    sns.histplot(df[col], kde=True, ax=ax, color='teal')
    ax.set_title(f"{col}\nskew={df[col].skew():.2f}")
fig.tight_layout()
plt.show()

print("Skewness of key numeric fields:")
print(df[key_metrics].skew().round(2))

**Note:** `Yield_Tonnes_Ha` and `Profit_INR` are both strongly right-skewed (skew > 2). This is expected — Sugarcane yields are an order of magnitude higher than grain crops, and a meaningful share of farms operate at a loss. This skew is why we rely on **medians and non-parametric statistical tests** rather than means and ANOVA later in this notebook.

## 5. Seasonal Performance Comparison — Yield & Production

**Key questions addressed:** *How does agricultural performance vary across seasons? What major seasonal patterns can be observed?*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Season', y='Yield_Tonnes_Ha', order=SEASON_ORDER,
            hue='Season', palette='Set2', legend=False, ax=axes[0])
axes[0].set_title('Raw Yield by Season (t/ha)')

sns.boxplot(data=df, x='Season', y='Yield_Zscore_within_Crop', order=SEASON_ORDER,
            hue='Season', palette='Set2', legend=False, ax=axes[1])
axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1].set_title('Crop-Normalized Yield by Season\n(z-score within each crop)')

fig.tight_layout()
plt.show()

print("Median yield by season (t/ha):")
print(df.groupby('Season', observed=True)['Yield_Tonnes_Ha'].median())
print("\nMean crop-normalized yield by season (0 = that crop's overall average):")
print(df.groupby('Season', observed=True)['Yield_Zscore_within_Crop'].mean().round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='Season', y='Production_Tonnes', order=SEASON_ORDER,
            hue='Season', palette='Set2', legend=False, ax=ax)
ax.set_yscale('log')
ax.set_title('Production by Season (log scale, tonnes)')
plt.show()

print("Total production by season (tonnes):")
print(df.groupby('Season', observed=True)['Production_Tonnes'].sum().round(0))

## 6. Environmental Conditions Across Seasons

**Key question addressed:** *Which characteristics change between seasons?*

A reusable helper function plots a grid of boxplots (one per variable), split by season, so we don't repeat the same plotting code for every group of columns.

In [ ]:
def season_boxplot_grid(data, cols, suptitle, ncols=3, figsize_per_plot=(5.2, 4.2)):
    """Plot a grid of boxplots (one per column in `cols`), grouped by Season."""
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per_plot[0]*ncols, figsize_per_plot[1]*nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, cols):
        sns.boxplot(data=data, x='Season', y=col, order=SEASON_ORDER,
                    hue='Season', palette='Set2', legend=False, ax=ax)
        ax.set_title(col.replace('_', ' '))
    for ax in axes[len(cols):]:
        ax.axis('off')
    fig.suptitle(suptitle, fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()

In [ ]:
env_cols = ['Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct',
            'Sunlight_Hours_Day', 'Soil_pH', 'Soil_Moisture_pct']

season_boxplot_grid(df, env_cols, 'Environmental Conditions by Season')

print("Mean environmental conditions by season:")
df.groupby('Season', observed=True)[env_cols].mean().round(2)

**Observed pattern:** Kharif (monsoon-fed) shows the highest rainfall, humidity and soil moisture; Zaid (summer, irrigation-dependent) shows the lowest rainfall/humidity but the highest temperature and sunlight hours. This matches the agronomic definition of these seasons and confirms the dataset is internally consistent.

## 7. Resource Usage Across Seasons

**Key questions addressed:** *Are there noticeable variations in resource usage across seasons? What differences exist between agricultural activities in different seasons?*

In [ ]:
resource_cols = ['Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Water_Used_m3',
                  'Nitrogen_kg_ha', 'Phosphorus_kg_ha', 'Potassium_kg_ha']

season_boxplot_grid(df, resource_cols, 'Resource Usage by Season')

print("Mean resource usage by season:")
df.groupby('Season', observed=True)[resource_cols].mean().round(2)

In [ ]:
irrigation_mix = pd.crosstab(df['Season'], df['Irrigation_Method'], normalize='index').loc[SEASON_ORDER] * 100

fig, ax = plt.subplots(figsize=(8, 5))
irrigation_mix.plot(kind='bar', stacked=True, colormap='viridis', ax=ax)
ax.set_ylabel('% of farms')
ax.set_title('Irrigation Method Mix by Season')
ax.legend(title='Irrigation Method', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("Irrigation method mix by season (%):")
irrigation_mix.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='Season', y='Water_Efficiency_t_per_1000m3', order=SEASON_ORDER,
            hue='Irrigation_Method', palette='Set2', ax=ax)
ax.set_title('Water Efficiency by Season and Irrigation Method')
ax.legend(title='Irrigation Method', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("Median water efficiency (t per 1000 m3) by season and irrigation method:")
df.pivot_table(values='Water_Efficiency_t_per_1000m3', index='Irrigation_Method',
                columns='Season', aggfunc='median', observed=True).round(2)

## 8. Relationships Between Environmental Conditions and Performance

**Key question addressed:** *Are there relationships between seasonal environmental conditions and agricultural performance?*

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).drop(columns=['Yield_Outlier'], errors='ignore').columns
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.3, ax=ax)
ax.set_title('Correlation Matrix — All Numeric Variables')
plt.tight_layout()
plt.show()

In [ ]:
factors = ['Rainfall_mm', 'Avg_Temperature_C', 'Soil_Moisture_pct', 'Fertilizer_kg_ha',
           'Nitrogen_kg_ha', 'Seed_Quality_Score']

season_corr = pd.DataFrame({
    season: df.loc[df['Season'] == season, factors].corrwith(df.loc[df['Season'] == season, 'Yield_Tonnes_Ha'])
    for season in SEASON_ORDER
}).round(3)

print("Correlation of each factor with Yield_Tonnes_Ha, computed separately within each season:")
season_corr

**Observed pattern:** correlations between individual environmental/input factors and yield are consistently weak (mostly under 0.1 in magnitude) *within* each season. This suggests that, in this dataset, seasonal yield differences are driven more by the **combination** of season-specific conditions and crop choice than by any single environmental variable acting alone — a useful finding to flag rather than force a stronger relationship than the data supports.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sample = df.sample(min(1000, len(df)), random_state=42)
sns.scatterplot(data=sample, x='Rainfall_mm', y='Yield_Tonnes_Ha', hue='Season',
                 hue_order=SEASON_ORDER, palette='Set2', alpha=0.6, ax=ax)
ax.set_yscale('log')
ax.set_title('Rainfall vs Yield by Season (log scale)')
plt.tight_layout()
plt.show()

## 9. Economic Outcomes Across Seasons

**Key question addressed:** *How do economic outcomes vary across seasons?*

In [ ]:
econ_summary = df.groupby('Season', observed=True).agg(
    Total_Revenue_INR=('Revenue_INR', 'sum'),
    Total_Cost_INR=('Total_Cost_INR', 'sum'),
    Total_Profit_INR=('Profit_INR', 'sum'),
    Avg_Profit_Margin_pct=('Profit_Margin_pct', 'mean'),
    Median_Profit_INR=('Profit_INR', 'median'),
    Loss_Making_Farms_pct=('Is_Loss_Making', lambda x: x.mean() * 100),
).loc[SEASON_ORDER]

econ_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

econ_summary[['Total_Revenue_INR', 'Total_Cost_INR']].plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title('Total Revenue vs Cost by Season')
axes[0].set_ylabel('INR')
axes[0].tick_params(axis='x', rotation=0)

sns.boxplot(data=df, x='Season', y='Profit_Margin_pct', order=SEASON_ORDER,
            hue='Season', palette='Set2', legend=False, ax=axes[1])
axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1].set_ylim(-200, 100)
axes[1].set_title('Profit Margin (%) by Season')

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=SEASON_ORDER, y=econ_summary['Loss_Making_Farms_pct'], hue=SEASON_ORDER,
            palette='Set2', legend=False, ax=ax)
ax.set_ylabel('% of farms operating at a loss')
ax.set_title('Loss-Making Farms by Season')
for i, v in enumerate(econ_summary['Loss_Making_Farms_pct']):
    ax.text(i, v + 1, f"{v:.1f}%", ha='center')
plt.tight_layout()
plt.show()

## 10. Regional & Crop-wise Consistency of Seasonal Patterns

**Key question addressed:** *Are some seasonal patterns consistent across different regions or categories?*

In [ ]:
pivot_state_yield = df.pivot_table(values='Yield_Tonnes_Ha', index='State', columns='Season',
                                     aggfunc='mean', observed=True)
pivot_crop_yield = df.pivot_table(values='Yield_Zscore_within_Crop', index='Crop', columns='Season',
                                    aggfunc='mean', observed=True)
pivot_state_profit = df.pivot_table(values='Profit_INR', index='State', columns='Season',
                                      aggfunc='mean', observed=True)

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
sns.heatmap(pivot_state_yield[SEASON_ORDER], annot=True, fmt='.1f', cmap='YlGnBu', ax=axes[0])
axes[0].set_title('Mean Yield (t/ha)\nby State x Season')

sns.heatmap(pivot_crop_yield[SEASON_ORDER], annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=axes[1])
axes[1].set_title('Mean Crop-Normalized Yield\n(z-score) by Crop x Season')

sns.heatmap(pivot_state_profit[SEASON_ORDER], annot=True, fmt=',.0f', cmap='RdYlGn', center=0, ax=axes[2])
axes[2].set_title('Mean Profit (INR)\nby State x Season')

fig.tight_layout()
plt.show()

In [ ]:
# Coefficient of variation (CV) across states, for each season — a measure of how
# consistent (low CV) or variable (high CV) the seasonal yield pattern is across regions
cv_by_season = (pivot_state_yield.std() / pivot_state_yield.mean() * 100).round(1)
print("Coefficient of variation of mean yield across states, per season (%):")
print(cv_by_season)
print("\nLower CV = the season's yield level is more consistent across states.")

**Observed pattern:** Zaid consistently shows the lowest average yield and worst economic outcomes across almost every state and crop, confirming this is a genuine seasonal effect rather than being driven by one region or crop.

## 11. Statistical Hypothesis Testing

**Key questions addressed:** *What differences exist between seasons, and are they statistically meaningful (not just due to chance)?*

Since `Yield_Tonnes_Ha` and `Profit_INR` are heavily right-skewed (see Section 4), we use the **Kruskal-Wallis test** (a non-parametric alternative to one-way ANOVA that compares ranks rather than means) instead of assuming normal distributions.

In [ ]:
def kruskal_by_season(data, col):
    groups = [g[col].dropna().values for _, g in data.groupby('Season', observed=True)]
    stat, p = stats.kruskal(*groups)
    return stat, p

for col in ['Yield_Tonnes_Ha', 'Profit_INR', 'Water_Efficiency_t_per_1000m3', 'Disease_Pest_Risk_pct']:
    stat, p = kruskal_by_season(df, col)
    sig = "significant" if p < 0.05 else "not significant"
    print(f"{col:32s}  H = {stat:8.2f}   p = {p:.4g}   -> {sig} difference across seasons")

Where the overall test is significant, we run **pairwise post-hoc tests** (Mann-Whitney U) between each pair of seasons, with a **Bonferroni correction** to control for the fact that we're running multiple comparisons.

In [ ]:
def pairwise_mannwhitney(data, col, seasons=SEASON_ORDER):
    pairs = list(combinations(seasons, 2))
    n_comparisons = len(pairs)
    results = []
    for a, b in pairs:
        ga = data.loc[data['Season'] == a, col]
        gb = data.loc[data['Season'] == b, col]
        u, p = stats.mannwhitneyu(ga, gb, alternative='two-sided')
        p_adj = min(p * n_comparisons, 1.0)
        results.append({'Comparison': f'{a} vs {b}', 'p_value': p, 'bonferroni_p': p_adj,
                         'significant (α=0.05)': p_adj < 0.05})
    return pd.DataFrame(results)

print("Pairwise comparison — Yield_Tonnes_Ha:")
display(pairwise_mannwhitney(df, 'Yield_Tonnes_Ha'))

print("\nPairwise comparison — Profit_INR:")
display(pairwise_mannwhitney(df, 'Profit_INR'))

We also test whether **categorical practices** (irrigation method, crop choice) differ by season, using the **Chi-square test of independence**.

In [ ]:
for cat_col in ['Irrigation_Method', 'Crop']:
    ct = pd.crosstab(df['Season'], df[cat_col])
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    sig = "significantly associated with" if p < 0.05 else "independent of"
    print(f"{cat_col:20s} is {sig} Season  (chi2={chi2:.2f}, dof={dof}, p={p:.4g})")

## 12. Unusual / Unexpected Pattern Detection

**Key question addressed:** *Are there unusual or unexpected seasonal patterns?*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x=SEASON_ORDER, y=df.groupby('Season', observed=True)['Disease_Pest_Risk_pct'].mean().loc[SEASON_ORDER],
            hue=SEASON_ORDER, palette='Set2', legend=False, ax=axes[0])
axes[0].set_ylabel('Mean Disease/Pest Risk (%)')
axes[0].set_title('Disease/Pest Risk by Season')

risk_threshold = df['Disease_Pest_Risk_pct'].quantile(0.90)
high_risk_counts = df[df['Disease_Pest_Risk_pct'] >= risk_threshold].groupby('Season', observed=True).size().loc[SEASON_ORDER]
sns.barplot(x=SEASON_ORDER, y=high_risk_counts, hue=SEASON_ORDER, palette='Set2', legend=False, ax=axes[1])
axes[1].set_ylabel(f'Farms above 90th percentile risk ({risk_threshold:.0f}%)')
axes[1].set_title('High-Risk Farm Count by Season')

fig.tight_layout()
plt.show()

print(f"{high_risk_counts.get('Kharif', 0)} of the top 10% riskiest farms are in Kharif season alone "
      f"— out of {int(len(df)*0.10)} total high-risk farms.")

**Unusual pattern:** Despite Kharif having the **highest total profit** of the three seasons (Section 9), it also carries by far the **highest disease/pest risk** and the largest concentration of high-risk farms. High rainfall/humidity conditions during Kharif appear to simultaneously boost potential production *and* pest/disease pressure — a genuine trade-off rather than a simple "better season" story.

In [ ]:
outlier_by_season = df.groupby('Season', observed=True)['Yield_Outlier'].agg(['sum', 'count'])
outlier_by_season['pct'] = (outlier_by_season['sum'] / outlier_by_season['count'] * 100).round(1)
print("Statistical yield outliers by season:")
print(outlier_by_season.loc[SEASON_ORDER])

In [ ]:
# Counter-intuitive cases: high rainfall but very low yield
high_rain_thresh = df['Rainfall_mm'].quantile(0.75)
low_yield_thresh = df['Yield_Tonnes_Ha'].quantile(0.25)

counter_intuitive = df[(df['Rainfall_mm'] >= high_rain_thresh) & (df['Yield_Tonnes_Ha'] <= low_yield_thresh)]
print(f"Farms with high rainfall (top 25%) but low yield (bottom 25%): {len(counter_intuitive)} "
      f"({len(counter_intuitive)/len(df)*100:.1f}% of all farms)")
print("\nBreakdown by season:")
print(counter_intuitive['Season'].value_counts())
print("\nBreakdown by crop:")
print(counter_intuitive['Crop'].value_counts())

## 13. Key Insights Summary

A consolidated, data-driven snapshot pulling together the findings from every section above.

In [ ]:
insights = []

# Best/worst season by yield and profit
best_yield_season = df.groupby('Season', observed=True)['Yield_Zscore_within_Crop'].mean().idxmax()
worst_yield_season = df.groupby('Season', observed=True)['Yield_Zscore_within_Crop'].mean().idxmin()
insights.append(f"Highest crop-normalized yield: {best_yield_season} | Lowest: {worst_yield_season}")

best_profit_season = econ_summary['Total_Profit_INR'].idxmax()
worst_profit_season = econ_summary['Total_Profit_INR'].idxmin()
insights.append(f"Highest total profit: {best_profit_season} | Lowest (or negative) total profit: {worst_profit_season}")

worst_loss_season = econ_summary['Loss_Making_Farms_pct'].idxmax()
insights.append(f"Highest share of loss-making farms: {worst_loss_season} "
                 f"({econ_summary.loc[worst_loss_season, 'Loss_Making_Farms_pct']:.1f}% of farms)")

riskiest_season = df.groupby('Season', observed=True)['Disease_Pest_Risk_pct'].mean().idxmax()
insights.append(f"Highest average disease/pest risk: {riskiest_season}")

wettest_season = df.groupby('Season', observed=True)['Rainfall_mm'].mean().idxmax()
driest_season = df.groupby('Season', observed=True)['Rainfall_mm'].mean().idxmin()
insights.append(f"Wettest season (avg rainfall): {wettest_season} | Driest: {driest_season}")

print("KEY INSIGHTS")
print("=" * 70)
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

## 14. Conclusions & Recommendations

### Conclusions

1. **Season has a real, statistically significant effect on both yield and profit** (Kruskal-Wallis p < 0.001 for both, confirmed pairwise between every season combination after Bonferroni correction) — this holds even after normalizing yield within each crop, so it isn't just a crop-mix artifact.

2. **Zaid is the weakest-performing season economically.** It has the lowest average rainfall and yield, the highest share of loss-making farms (roughly two-thirds), and the only *negative* total profit of the three seasons — despite having the smallest sample of farms overall (i.e., this isn't a small-sample fluke, the pattern is consistent across states and crops).

3. **Kharif carries a "high risk, high reward" profile.** It has the highest total profit of the three seasons, but also the highest average disease/pest risk and the largest concentration of high-risk farms — most likely driven by the high humidity and rainfall that also support stronger growth. This is a genuine trade-off, not simply "the best season."

4. **Individual environmental factors are weak predictors of yield on their own.** Within any given season, rainfall, temperature, soil moisture and fertilizer use each correlate only weakly with yield (|r| mostly < 0.1). Season-level averages differ a lot, but at the individual-farm level, yield is evidently shaped by a combination of factors (crop choice, input quality, management) rather than any single environmental driver.

5. **Farming practices are not meaningfully adapted by season.** Both irrigation method mix and crop mix are statistically independent of season (chi-square tests not significant), meaning farmers largely use the same irrigation methods and grow similar crop proportions regardless of season — despite Section 6-7 showing that seasonal conditions (rainfall, temperature, soil moisture) differ substantially.

### Recommendations

- **Prioritize Zaid-season support.** Given its high loss-making rate and negative aggregate profit, targeted interventions — subsidized irrigation, crop insurance, or promoting lower-water-requirement crop varieties — would likely have the largest impact per rupee invested.
- **Strengthen pest/disease management specifically for Kharif.** Since high disease/pest risk coincides with the most profitable season, investment in disease-resistant seed varieties and monsoon-specific pest advisories could protect (and likely grow) Kharif's profit advantage.
- **Encourage season-specific irrigation choices.** Since irrigation method doesn't currently vary by season despite very different rainfall levels, promoting drip/sprinkler irrigation more heavily in the drier Zaid season (where water efficiency matters most) is a concrete, low-cost lever.
- **Investigate input-efficiency rather than raw environmental conditions.** Because weather/soil factors alone explain little of the yield variation, further analysis should look at fertilizer *efficiency* (yield per kg applied), seed quality, and farm management practices as the more promising levers for improving performance.
- **Use crop-normalized metrics for any future seasonal comparison.** Raw yield comparisons across seasons can be misleading if crop mix shifts even slightly; the z-score-within-crop approach used in this notebook should be the default for season-level reporting going forward.

*(These conclusions are generated directly from the dataset analyzed above — re-run this notebook if the dataset is updated, as the specific figures may change.)*